# fraud-v3-candidate — shadow run investigation

The model platform blocked the promotion (TESS-2310).

| | AUC | PR-AUC |
|---|---|---|
| candidate, offline (notebook 01) | **0.906** | 0.358 |
| candidate, shadow (2026-07-05 → 2026-08-27) | **0.699** | 0.071 |
| champion fraud-v2, same window | 0.778 | 0.164 |

Offline it beat the champion by a wide margin. In shadow it is *worse* than the champion. Why?

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
print(ROOT)

/Users/vishal/Documents/GitHub/fintech1/tessera-platform


In [2]:
report = json.loads((ROOT / "ml/registry/shadow/fraud-v3-candidate.json").read_text())
pd.DataFrame({
    "offline": report["offline"],
    "shadow": {k: v for k, v in report["shadow"].items() if k != "by_month"},
    "champion": report["champion_same_window"],
}).T

,auc,pr_auc,source
offline,0.9059,0.3578,"ml/notebooks/01_fraud_exploration.ipynb, cell 6"
shadow,0.6992,0.0706,NaN
champion,0.7782,0.164,NaN


In [3]:
df = pd.read_parquet(ROOT / "ml/data/transactions.parquet").sort_values("timestamp").reset_index(drop=True)
cut = df.timestamp.quantile(0.70)          # same temporal split as notebook 01
train, test = df[df.timestamp <= cut].copy(), df[df.timestamp > cut].copy()
print(f"train={len(train):,}  test={len(test):,}  test window {test.timestamp.min().date()} → {test.timestamp.max().date()}")
print("the shadow window is the test window")

train=42,000  test=18,000  test window 2026-07-05 → 2026-08-27
the shadow window is the test window


## Questions

1. Can the offline number be reproduced from notebook 01's feature set?
2. The report says `card_chargeback_rate` is non-zero for 27.4% of training rows but 14.2% of scored rows. Does the feature look the same at training time as at scoring time?
3. What does the candidate score without it?

## 1. Reproduce notebook 01

Same table, same 70/30 temporal split, same code as notebook 01 cells 6 and 8. If this does not land on 0.906, the rest of this notebook is about the wrong model.

In [4]:
import sys
sys.path.insert(0, str(ROOT))  # the productionised modules under ml/ are imported further down

# --- verbatim from notebook 01 -------------------------------------------
dev_counts = df.groupby("device_id").card_token.nunique()

def base_features(d):
    return pd.DataFrame({
        "amount_log": np.log1p(d.amount_minor),
        "hour": d.timestamp.dt.hour,
        "is_night": ((d.timestamp.dt.hour >= 1) & (d.timestamp.dt.hour <= 5)).astype(int),
        "is_cross_border": d.is_cross_border,
        "mcc": d.mcc.astype("category").cat.codes,
        "device_card_count": d.device_id.map(dev_counts).fillna(1),
    }, index=d.index)

def evaluate(Xtr, Xte, label):
    m = HistGradientBoostingClassifier(max_iter=250, random_state=0).fit(Xtr, train.is_fraud)
    p = m.predict_proba(Xte)[:, 1]
    auc = roc_auc_score(test.is_fraud, p)
    ap = average_precision_score(test.is_fraud, p)
    print(f"{label:<50} AUC={auc:.4f}  PR-AUC={ap:.4f}")
    return m, p, auc, ap

# notebook 01 cell 8: one rate per card, over the whole table
card_cb_rate_full = df.groupby("card_token").chargeback_filed_at.apply(lambda s: s.notna().mean())

def with_card_history(d):
    X = base_features(d)
    X["card_chargeback_rate"] = d.card_token.map(card_cb_rate_full).fillna(0.0)
    return X
# -------------------------------------------------------------------------

baseline_model, baseline_p, baseline_auc, baseline_ap = evaluate(
    base_features(train), base_features(test), "baseline (v2 feature set)")
nb01_model, nb01_p, nb01_auc, nb01_ap = evaluate(
    with_card_history(train), with_card_history(test), "+ card_chargeback_rate (notebook 01 definition)")

assert abs(nb01_auc - report["offline"]["auc"]) < 5e-4, "does not reproduce the offline number"
assert abs(baseline_auc - report["champion_same_window"]["auc"]) < 5e-4, "does not reproduce the champion"
print("\nreproduced: 0.906 is the notebook-01 definition; the baseline is the champion's 0.778 on the same window")

baseline (v2 feature set)                          AUC=0.7782  PR-AUC=0.1640
+ card_chargeback_rate (notebook 01 definition)    AUC=0.9059  PR-AUC=0.3578

reproduced: 0.906 is the notebook-01 definition; the baseline is the champion's 0.778 on the same window


## 2. Does the feature look the same at training time as at scoring time?

No. Notebook 01 computes **one number per card over the whole table** and maps it onto every row of that card. For a transaction on day *D* that number counts every dispute ever filed against the card: disputes filed after *D*, and the dispute of *D itself*, which is filed 20–90 days later (median 54 in this data). At the moment of scoring on day *D*, none of those exist yet.

The serving path (`card_history`) counts only chargebacks **filed as of scoring time**. Training and serving computed two different features under one name.

`ml/features/aggregates.card_chargeback_rate(df, as_of)` is the serving definition made explicit: per row, chargebacks filed before `as_of` over transactions before `as_of`. Applied to the training table with `as_of` = each transaction's own timestamp, it is what the model *should* have been trained on.

In [5]:
from ml.features import aggregates

cb_full = df.card_token.map(card_cb_rate_full).fillna(0.0)                       # notebook 01
cb_pit = aggregates.card_chargeback_rate(df, aggregates.scoring_time_as_of(df))  # point-in-time

reported = report["feature_stats"]["card_chargeback_rate"]
pd.DataFrame({
    "non-zero share": [(cb_full[train.index] > 0).mean(), (cb_pit[train.index] > 0).mean(), (cb_pit[test.index] > 0).mean()],
    "mean": [cb_full[train.index].mean(), cb_pit[train.index].mean(), cb_pit[test.index].mean()],
    "reported in shadow report": [reported["training_nonzero_share"], np.nan, reported["shadow_nonzero_share"]],
}, index=[
    "training rows, notebook-01 definition (what the model was trained on)",
    "training rows, point-in-time (what serving would have produced)",
    "shadow window, point-in-time (what serving did produce)",
]).round(4)

,non-zero share,mean,reported in shadow report
"training rows, notebook-01 definition (what the model was trained on)",0.2743,0.0297,0.2743
"training rows, point-in-time (what serving would have produced)",0.0344,0.0067,NaN
"shadow window, point-in-time (what serving did produce)",0.1419,0.0176,0.1419


Both reported numbers reproduce to four decimals: **27.4%** of training rows were non-zero under the notebook definition, and **14.2%** of shadow rows were non-zero under the serving definition. The same serving definition applied to the training rows gives 3.4%: the training table had barely accumulated any *filed* dispute history yet, because the synthetic window starts cold on 2026-03-01 and disputes take 20–90 days to arrive.

Where do the notebook definition's non-zero values come from?

In [6]:
hist = aggregates.card_history(df, aggregates.scoring_time_as_of(df))   # prior_transactions, prior_chargebacks per row
own_disputed = df.chargeback_filed_at.notna()
on_disputed_card = cb_full > 0

print(f"rows whose card has >= 1 dispute anywhere in the table (non-zero under notebook 01): {on_disputed_card.mean():.1%}")
print(f"  of which, none of the card's disputes had been filed yet at the transaction's timestamp: "
      f"{(hist.prior_chargebacks[on_disputed_card] == 0).mean():.1%}")
print(f"  of which, the row's OWN transaction is one of the disputed ones:                        "
      f"{own_disputed[on_disputed_card].mean():.1%}")
print(f"\nfraud rows in the shadow window whose own chargeback is in the table: "
      f"{own_disputed[test.index][test.is_fraud == 1].mean():.1%}   (the generator disputes ~85% of fraud)")

rows whose card has >= 1 dispute anywhere in the table (non-zero under notebook 01): 27.4%
  of which, none of the card's disputes had been filed yet at the transaction's timestamp: 75.7%
  of which, the row's OWN transaction is one of the disputed ones:                        10.8%

fraud rows in the shadow window whose own chargeback is in the table: 86.1%   (the generator disputes ~85% of fraud)


Three-quarters of the rows the notebook definition marks as "on a disputed card" had **no filed dispute at all** at the time of the transaction. So how much of the feature's ranking power is the row's own future chargeback? Score the shadow window with the feature alone, under three definitions.

In [7]:
# leave-own-row-out: the notebook definition minus the row's own outcome, still over the whole table
n_card = df.groupby("card_token").chargeback_filed_at.transform("size")
k_card = df.groupby("card_token").chargeback_filed_at.transform("count")
cb_loo = ((k_card - own_disputed.astype(int)) / (n_card - 1).where(n_card > 1)).fillna(0.0)

pd.DataFrame({
    "single-feature AUC, shadow window": [
        roc_auc_score(test.is_fraud, cb_full[test.index]),
        roc_auc_score(test.is_fraud, cb_loo[test.index]),
        roc_auc_score(test.is_fraud, cb_pit[test.index]),
    ]
}, index=["notebook 01 (whole table, own row included)", "whole table, own row excluded", "point-in-time"]).round(4)

,"single-feature AUC, shadow window"
"notebook 01 (whole table, own row included)",0.8481
"whole table, own row excluded",0.5116
point-in-time,0.4976


**0.85 → 0.51 → 0.50.** Remove the row's own chargeback and the feature is a coin flip; make it point-in-time and it is exactly a coin flip. The entire lift in notebook 01 was the model reading each transaction's own dispute, filed weeks after the transaction, back through a per-card average.

The correlation of 0.285 that notebook 01 called "the strongest single feature we've ever had" is the correlation between *being disputed* and *being fraud*. That is not a feature. It is 85% of the label.

## 3. Reproducing the shadow number

If the story above is right, the shadow result is simply the notebook-01 model, trained on the whole-table feature, being handed the point-in-time feature by the serving path. No retraining: the same fitted object from section 1.

In [8]:
def with_feature(d, feature):
    X = base_features(d)
    X["card_chargeback_rate"] = feature.loc[d.index].to_numpy()
    return X

p_shadow = nb01_model.predict_proba(with_feature(test, cb_pit))[:, 1]
month = test.timestamp.dt.strftime("%Y-%m")

reproduced = {
    "AUC": roc_auc_score(test.is_fraud, p_shadow),
    "PR-AUC": average_precision_score(test.is_fraud, p_shadow),
    **{f"AUC {m}": roc_auc_score(test.is_fraud[month == m], p_shadow[(month == m).to_numpy()]) for m in sorted(month.unique())},
}
shadow = {"AUC": report["shadow"]["auc"], "PR-AUC": report["shadow"]["pr_auc"],
          **{f"AUC {m['month']}": m["auc"] for m in report["shadow"]["by_month"]}}
pd.DataFrame({"shadow report": shadow, "reproduced here": reproduced}).round(4)

,shadow report,reproduced here
AUC,0.6992,0.6992
PR-AUC,0.0706,0.0706
AUC 2026-07,0.7060,0.7060
AUC 2026-08,0.6950,0.6946


All four numbers reproduce to the precision the report gives them: **0.699 / 0.071**, and 0.706 / 0.695 by month. The shadow run did nothing wrong. It computed the feature the only way a serving path can, and the model it was given had been trained on something else.

One more check pins down *exactly* what serving computed. The report says the feature comes from a nightly batch. If the batch were a day stale, the effective cutoff would be midnight rather than the scoring timestamp:

In [9]:
cb_nightly = aggregates.card_chargeback_rate(df, aggregates.nightly_batch_as_of(df))
p_nightly = nb01_model.predict_proba(with_feature(test, cb_nightly))[:, 1]
pd.DataFrame({
    "as of scoring timestamp": [(cb_pit[test.index] > 0).mean(), roc_auc_score(test.is_fraud, p_shadow)],
    "as of midnight (day-stale batch)": [(cb_nightly[test.index] > 0).mean(), roc_auc_score(test.is_fraud, p_nightly)],
    "shadow report": [reported["shadow_nonzero_share"], report["shadow"]["auc"]],
}, index=["non-zero share, shadow window", "AUC, shadow window"]).round(4)

,as of scoring timestamp,as of midnight (day-stale batch),shadow report
"non-zero share, shadow window",0.1419,0.1411,0.1419
"AUC, shadow window",0.6992,0.6997,0.6992


The scoring-timestamp cutoff matches the report exactly (0.1419, 0.6992); the midnight cutoff is close but not it (0.1411, 0.6997). So serving is fresh to the transaction, and that is the cutoff `ml/fraud/pipeline.py` now trains with. The difference between the two is 0.0005 AUC: batch staleness is not the gap, and was never going to be.

## 4. Why *worse* than the champion, not merely no better?

A useless feature should cost roughly nothing. This one cost 0.08 AUC against the baseline. The model did not learn "this feature is noise"; it learned "this feature is the answer", and at serving time it kept believing that. Split the shadow window by what the model actually saw.

In [10]:
seen = pd.DataFrame({
    "feature at serving": np.where(cb_pit[test.index] > 0, "non-zero", "zero"),
    "fraud": test.is_fraud.to_numpy(),
    "nb01 model": p_shadow,
    "baseline model": baseline_p,
}, index=test.index)

rows = []
for value, g in seen.groupby("feature at serving"):
    rows.append({
        "feature at serving": value,
        "rows": len(g),
        "share of traffic": len(g) / len(seen),
        "fraud rate": g.fraud.mean(),
        "mean score, nb01 model": g["nb01 model"].mean(),
        "mean score, baseline": g["baseline model"].mean(),
        "AUC within group, nb01": roc_auc_score(g.fraud, g["nb01 model"]),
        "AUC within group, baseline": roc_auc_score(g.fraud, g["baseline model"]),
    })
pd.DataFrame(rows).set_index("feature at serving").round(4)

,rows,share of traffic,fraud rate,"mean score, nb01 model","mean score, baseline","AUC within group, nb01","AUC within group, baseline"
feature at serving,,,,,,,
non-zero,2554,0.1419,0.0345,0.1112,0.0350,0.7315,0.7620
zero,15446,0.8581,0.0354,0.0069,0.0356,0.7501,0.7809


In [11]:
# What the review queue would have looked like, at each model's own top-5% threshold on its training scores.
thr_nb01 = np.quantile(nb01_model.predict_proba(with_card_history(train))[:, 1], 0.95)
thr_base = np.quantile(baseline_model.predict_proba(base_features(train))[:, 1], 0.95)
nonzero = (cb_pit[test.index] > 0).to_numpy()
y = test.is_fraud.to_numpy()
flag_nb01, flag_base = p_shadow >= thr_nb01, baseline_p >= thr_base

print(f"notebook-01 model in shadow, threshold {thr_nb01:.3f}:")
print(f"  flags {flag_nb01.mean():.1%} of traffic; {nonzero[flag_nb01].mean():.0%} of the queue is rows whose feature was non-zero")
print(f"  precision on flagged rows with non-zero feature: {y[flag_nb01 & nonzero].mean():.3f}")
print(f"  precision on flagged rows with zero feature    : {y[flag_nb01 & ~nonzero].mean():.3f}")
print(f"  recall {flag_nb01[y == 1].mean():.3f}, precision {y[flag_nb01].mean():.3f}")
print(f"baseline model, threshold {thr_base:.3f}:")
print(f"  flags {flag_base.mean():.1%} of traffic; recall {flag_base[y == 1].mean():.3f}, precision {y[flag_base].mean():.3f}")

from ml.validation.drift import psi
print(f"\nPSI of the feature the model was trained on vs the feature it was served: "
      f"{psi(cb_full[train.index], cb_pit[test.index]).value:.3f}")

notebook-01 model in shadow, threshold 0.184:
  flags 2.6% of traffic; 98% of the queue is rows whose feature was non-zero
  precision on flagged rows with non-zero feature: 0.092
  precision on flagged rows with zero feature    : 0.111
  recall 0.068, precision 0.092
baseline model, threshold 0.122:
  flags 5.1% of traffic; recall 0.277, precision 0.191

PSI of the feature the model was trained on vs the feature it was served: 0.151


Two things go wrong at once, and both are in the table:

1. **Non-zero means "confirmed dispute" to the model, but "some old dispute on this card" to the world.** The 14% of shadow rows with a non-zero feature have the same fraud rate as everyone else (3.5%), yet the notebook-01 model scores them sixteen times higher on average. At its own review threshold, 98% of its queue is these rows, and its precision on them is 0.09 against the baseline's 0.19.
2. **Zero means "safe" to the model.** Within the 86% of traffic where the feature is zero, the notebook-01 model ranks fraud *worse* than the baseline does (AUC 0.750 vs 0.781), because it learned to lean on the feature instead of on the observable ones. The observable features did not stop working; the model stopped using them.

The practical result in shadow: the model fills only half its review budget (2.6% of traffic against 5%) and recall falls from 0.28 to 0.07. Had it been promoted, it would have caught a quarter of the fraud the champion catches. The feature it was trained on and the feature it was served are different distributions (PSI 0.15), which is the train/serve skew Gate 3 now measures.

## 5. What does the candidate score without the leak?

Retrain with the point-in-time feature in both training *and* holdout (the model the shadow run should have been given), and compare with the production pipeline's feature set (`ml/features/base.py` + the point-in-time aggregate), which is what `ml/fraud/pipeline.py` now defines as fraud-v3-candidate.

In [12]:
_, _, pit_auc, pit_ap = evaluate(with_feature(train, cb_pit), with_feature(test, cb_pit), "+ card_chargeback_rate (point-in-time)")
print(f"{'lift over baseline':<50} {pit_auc - baseline_auc:+.4f}      {pit_ap - baseline_ap:+.4f}")

from ml.fraud import pipeline
from ml.validation import performance
pipe_df = pipeline.load_transactions()   # the table as the pipeline loads it -- CI's gates see exactly this
gate2 = performance.evaluate(pipe_df, min_pr_auc=0.15)
print()
print(f"{'pipeline candidate (base.py + point-in-time)':<50} AUC={gate2.overall.auc:.4f}  PR-AUC={gate2.overall.pr_auc:.4f}  recall@5%={gate2.overall.recall:.3f}")
print(f"{'pipeline champion (base.py)':<50} AUC={gate2.champion.auc:.4f}  PR-AUC={gate2.champion.pr_auc:.4f}  recall@5%={gate2.champion.recall:.3f}")
for m in gate2.by_month:
    print(f"{'  candidate, ' + m.label:<50} AUC={m.auc:.4f}  PR-AUC={m.pr_auc:.4f}")
print(f"\nGate 2 at the workflow floor (PR-AUC >= 0.15): {'PASS' if gate2.ok else 'FAIL'}; advisories: {gate2.advisories}")

+ card_chargeback_rate (point-in-time)             AUC=0.7775  PR-AUC=0.1562
lift over baseline                                 -0.0008      -0.0078



pipeline candidate (base.py + point-in-time)       AUC=0.7788  PR-AUC=0.1590  recall@5%=0.283
pipeline champion (base.py)                        AUC=0.7802  PR-AUC=0.1602  recall@5%=0.285
  candidate, 2026-07                               AUC=0.7683  PR-AUC=0.1541
  candidate, 2026-08                               AUC=0.7900  PR-AUC=0.1663

Gate 2 at the workflow floor (PR-AUC >= 0.15): PASS; advisories: ['candidate does not beat the champion on PR-AUC (0.1590 vs 0.1602) -- clearing the floor is not a reason to promote', 'candidate does not beat the champion on AUC (0.7788 vs 0.7802)']


Point-in-time, the candidate **is the champion**: 0.778 AUC either way, PR-AUC within noise of 0.16. The feature adds nothing. Not "less than hoped"; nothing. (This notebook's loader and the pipeline's order tied timestamps differently, which moves the third decimal and flips the sign of the candidate-vs-champion PR-AUC gap; the pipeline numbers above are what CI's Gate 2 prints. A gap whose sign depends on row order is not a gap.)

That is not surprising once you look at how the data is made. In `ml/data/generate.py` each transaction draws its card uniformly at random; fraud propensity lives on the merchant, the MCC, the hour, the amount, cross-border and the device, and nothing about a card's past predicts its future. A real portfolio may behave differently (a card that has been disputed before may genuinely be riskier), but that is now a hypothesis to test with the honest definition, not a result anyone has seen.

### The other full-table aggregate

`device_card_count` in notebook 01 is also a `groupby` over the whole table: it counts every card that will *ever* use the device, including ones that first appear months later. It reads no outcome column, so it is not target leakage, but it is still the future. How much does it matter?

In [13]:
pairs = df[["device_id", "card_token", "timestamp"]].copy()
first_seen = pairs.groupby(["device_id", "card_token"]).timestamp.transform("min")
pairs["new_card_on_device"] = (pairs.timestamp == first_seen).astype(int)
dev_cards_pit = pairs.groupby("device_id").new_card_on_device.cumsum()   # df is time-ordered: cards seen on the device so far

def with_pit_device(d, feature=None):
    X = base_features(d)
    X["device_card_count"] = dev_cards_pit.loc[d.index].to_numpy()
    if feature is not None:
        X["card_chargeback_rate"] = feature.loc[d.index].to_numpy()
    return X

evaluate(with_pit_device(train), with_pit_device(test), "baseline, point-in-time device_card_count")
evaluate(with_pit_device(train, cb_pit), with_pit_device(test, cb_pit), "  + point-in-time card_chargeback_rate");

baseline, point-in-time device_card_count          AUC=0.7739  PR-AUC=0.1630


  + point-in-time card_chargeback_rate             AUC=0.7723  PR-AUC=0.1601


About 0.004 AUC. Real, small, and present in the champion too: fraud-v2 serves `device_card_count` somehow, and *how* is a question for platform (follow-up in the PR). Gate 1 reports this class of feature as an advisory rather than a failure, which is the right severity. It changes the number a little; the chargeback rate changed it entirely.

## 6. Gate 1 on both definitions

`ml/validation/leakage.py` is section 2 productionised: it recomputes every feature with future disputes hidden, then with each row's own dispute hidden, and fails if any value moves. Run it on the notebook definition and on the pipeline's.

In [14]:
from ml.validation import leakage

def notebook01_definition(frame):   # exactly notebook 01, cell 8, as a feature function
    rate = frame.groupby("card_token").chargeback_filed_at.apply(lambda s: s.notna().mean())
    return frame.card_token.map(rate).fillna(0.0)

pipeline_fns = pipeline.feature_functions()
notebook_fns = {**pipeline_fns, "card_chargeback_rate": notebook01_definition}

for label, fns in [("notebook 01 definition", notebook_fns), ("pipeline (point-in-time) definition", pipeline_fns)]:
    r = leakage.run_probes(df, fns, own_outcome_sample=5)
    verdict = ("LEAK in " + ", ".join(r.leaking_features)) if not r.ok else "no leak detected"
    print(f"{label:<40} {verdict}")
    for probe in r.leaks:
        print(f"    {probe.probe:<24} {probe.cutoff:<18} {probe.rows_changed:>6,} of {probe.rows_compared:>6,} rows changed, max |change| {probe.max_abs_change:.3f}")
    for w in r.warnings:
        print(f"    advisory: {w.feature} changes when future rows are removed ({w.probe}, {w.cutoff}) -- frame-wide statistic, not target leakage")
    for line in r.advisories:
        print(f"    advisory: {line}")

notebook 01 definition                   LEAK in card_chargeback_rate
    future outcomes hidden   T=2026-04-14        4,000 of 15,000 rows changed, max |change| 0.500
    future outcomes hidden   T=2026-05-29        6,708 of 30,000 rows changed, max |change| 0.400
    future outcomes hidden   T=2026-07-13        7,188 of 45,000 rows changed, max |change| 0.400
    own outcome hidden       5 disputed rows         5 of      5 rows changed, max |change| 0.200
    advisory: amount_zscore_within_mcc changes when future rows are removed (future rows removed, T=2026-04-14) -- frame-wide statistic, not target leakage
    advisory: amount_zscore_within_mcc changes when future rows are removed (future rows removed, T=2026-05-29) -- frame-wide statistic, not target leakage
    advisory: amount_zscore_within_mcc changes when future rows are removed (future rows removed, T=2026-07-13) -- frame-wide statistic, not target leakage
    advisory: `card_chargeback_rate` alone ranks holdout fraud at AUC 

pipeline (point-in-time) definition      no leak detected
    advisory: amount_zscore_within_mcc changes when future rows are removed (future rows removed, T=2026-04-14) -- frame-wide statistic, not target leakage
    advisory: amount_zscore_within_mcc changes when future rows are removed (future rows removed, T=2026-05-29) -- frame-wide statistic, not target leakage
    advisory: amount_zscore_within_mcc changes when future rows are removed (future rows removed, T=2026-07-13) -- frame-wide statistic, not target leakage


## 7. Gate 3: the honest feature drifts, for a reason the reviewer needs to know

The point-in-time feature passes Gate 1, but Gate 3 reports it at PSI 0.25, exactly at the workflow limit. That number is real, and it is not the population moving.

In [15]:
from ml.validation import drift

gate3 = drift.evaluate(pipe_df, max_psi=0.25)
row = gate3.row("card_chargeback_rate")
print(f"Gate 3 (train vs holdout): card_chargeback_rate PSI={row.psi.value:.4f} over {row.psi.n_bins} bins -> "
      f"{row.status(gate3.warn_psi, gate3.max_psi)};  non-zero share {row.train_nonzero:.3f} -> {row.recent_nonzero:.3f}\n")

cb_pipe = pipeline.build_features(pipe_df)["card_chargeback_rate"]
pipe_month = pipe_df.timestamp.dt.strftime("%Y-%m")
print(pd.DataFrame({
    "non-zero share": (cb_pipe > 0).groupby(pipe_month).mean(),
    "mean prior transactions on the card": aggregates.card_history(
        pipe_df, aggregates.scoring_time_as_of(pipe_df)).prior_transactions.groupby(pipe_month).mean(),
}).round(3), "\n")

split = pipeline.temporal_split(pipe_df)
for since in ["2026-03-01", "2026-04-01", "2026-05-01", "2026-06-01"]:
    tr = split.train[(pipe_df.loc[split.train, "timestamp"] >= pd.Timestamp(since, tz="UTC")).to_numpy()]
    print(f"training rows from {since} (n={len(tr):>6,}) vs holdout: PSI={drift.psi(cb_pipe[tr], cb_pipe[split.test]).value:.4f}")
jul, aug = (pipe_df.index[pipe_month == m] for m in ("2026-07", "2026-08"))
print(f"July vs August (both inside the holdout):               PSI={drift.psi(cb_pipe[jul], cb_pipe[aug]).value:.4f}")

Gate 3 (train vs holdout): card_chargeback_rate PSI=0.2499 over 11 bins -> warn;  non-zero share 0.034 -> 0.142



           non-zero share  mean prior transactions on the card
timestamp                                                     
2026-03             0.000                                0.860
2026-04             0.010                                2.581
2026-05             0.038                                4.257
2026-06             0.081                                5.917
2026-07             0.118                                7.653
2026-08             0.164                                9.231 

training rows from 2026-03-01 (n=42,000) vs holdout: PSI=0.2499
training rows from 2026-04-01 (n=31,612) vs holdout: PSI=0.2095
training rows from 2026-05-01 (n=21,607) vs holdout: PSI=0.1603
training rows from 2026-06-01 (n=11,372) vs holdout: PSI=0.1049
July vs August (both inside the holdout):               PSI=0.0522


The table starts on 2026-03-01 with no history at all: no card has a prior transaction, let alone a filed dispute, so the feature is identically zero for the whole first month and its non-zero share climbs every month after that as history accumulates. The PSI falls monotonically as the cold start is trimmed from the training side, and even July against August is still moving. This is **left-censoring of the synthetic table**, not a change in the population. In production, card histories go back years and the feature should be stationary.

"Should be" is not a measurement. Before this feature is trusted on real traffic, Gate 3 has to be run against a training extract with mature history, and if the feature is still moving the gate will say so. The model card records this as the first open item.

## Conclusion

| | AUC | PR-AUC | what it is |
|---|---|---|---|
| notebook 01, offline | 0.906 | 0.358 | model reads each transaction's own future chargeback through a per-card average |
| shadow, as reported | 0.699 | 0.071 | the same model, handed the feature as it actually exists at scoring time |
| shadow, reproduced here | 0.699 | 0.071 | same fitted object, point-in-time feature; by month 0.706 / 0.695 |
| candidate, point-in-time in training *and* serving | 0.778 | 0.156 | the honest candidate, notebook feature set |
| pipeline candidate, CI Gate 2 | 0.779 | 0.159 | `base.py` features + point-in-time aggregate, as `ml/fraud/pipeline.py` defines it |
| champion fraud-v2 | 0.778 | 0.164 | the baseline |

**Root cause.** `card_chargeback_rate` was computed as one number per card over the whole training table (notebook 01, cell 8). For every training row it therefore included the row's own chargeback, filed 20–90 days after the transaction. At serving time that dispute does not exist. Training and serving computed different features under one name: the offline number measured the leak, and the shadow number measured a model that had learned to depend on it.

**Why worse than the champion.** The model treats a non-zero feature as near-certain fraud, but at serving time non-zero means "a card with some old dispute", which carries no signal; and within the 86% of traffic where the feature is zero it ranks worse than the baseline, because it leaned on the feature instead of the observable ones.

**Honest lift: none.** In this data a card's past disputes do not predict its future; the generator gives cards no persistent risk. That may differ on real traffic, and can now be tested honestly.

**Ruled out**, with numbers:
- *Drift between training and shadow windows.* PSI on every base feature < 0.002; the champion scores 0.778 in both. The population did not move.
- *Label immaturity in shadow* (labels as of 2026-09-15, window ends 2026-08-27). The champion is unaffected, both shadow months are equally bad, and the fully-labelled reproduction lands on 0.6992 without any censoring.
- *Nightly-batch staleness.* Midnight vs scoring-timestamp cutoffs differ by 0.0005 AUC, and the serving stats match the scoring-timestamp cutoff exactly.
- *A serving bug.* The serving feature's non-zero share (14.19%) matches the honest definition to four decimals. Serving did the right thing.
- *A different model in shadow.* One fitted object reproduces all four shadow numbers.

**Confidence: high.** Four independent shadow numbers reproduce to four decimals from one mechanism.

**Recommendation.** Do not promote fraud-v3-candidate. Its registry entry's 0.906 is not a number this model can produce. The productionised pieces in this PR (`ml/features/aggregates.py`, `ml/fraud/pipeline.py`, the four gates under `ml/validation/`, and the model card) make the honest number the only one the pipeline can compute, and make this mistake fail CI the next time it is made.